# Energia, Potência e Densidade Espectral
## Seções 3.7 e 3.8 — Curso de Princípios de Comunicação — UnB

**Objetivo:** Visualizar de forma interativa os conceitos de energia/potência de sinais,  
Teorema de Parseval, DEE (Densidade Espectral de Energia), PSD (Density Espectral de Potência),  
Teorema de Wiener-Khinchin e efeitos de ruído em sistemas de comunicação.

---
| Seção | Conteúdo |
|---|---|
| 1 | Classificação: sinais de energia vs. sinais de potência |
| 2 | Teorema de Parseval — verificação numérica |
| 3 | DEE: distribuição espectral de energia |
| 4 | Largura de banda que contém X% da energia |
| 5 | Princípio da Incerteza (Heisenberg-Gabor) |
| 6 | PSD de sinais periódicos |
| 7 | Teorema de Wiener-Khinchin |
| 8 | Ruído branco e filtragem — SNR |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal as sig
from scipy.fft import fft, ifft, fftfreq, fftshift

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.35,
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
})
print('Pacotes carregados com sucesso.')

---
## 1. Classificação: Sinal de Energia vs. Sinal de Potência

Um sinal é **de energia** se $0 < E < \infty$ (e portanto $P = 0$).  
Um sinal é **de potência** se $0 < P < \infty$ (e portanto $E = \infty$).

$$E = \int_{-\infty}^{\infty} |f(t)|^2\,dt \qquad P = \lim_{T\to\infty}\frac{1}{T}\int_{-T/2}^{T/2}|f(t)|^2\,dt$$

In [ ]:
fs = 1000
T_obs = 10
t = np.arange(-T_obs/2, T_obs/2, 1/fs)

# --- Sinais ---
tau = 2.0
pulso = np.where(np.abs(t) <= tau/2, 1.0, 0.0)   # pulso retangular — energia finita
exp_dec = np.exp(-np.abs(t))                        # exponencial bilateral — energia finita
cosseno = np.cos(2*np.pi*2*t)                       # senoide — energia INFINITA (potência finita)

# --- Energia numérica (integração) ---
dt = 1/fs
E_pulso = np.sum(pulso**2) * dt
E_exp   = np.sum(exp_dec**2) * dt
E_cos   = np.sum(cosseno**2) * dt  # cresce com T_obs
P_cos   = E_cos / T_obs

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sinais = [(pulso, f'Pulso retangular ($\\tau={tau}$ s)', f'E = {E_pulso:.2f} J  |  P ≈ 0', 'royalblue'),
          (exp_dec, 'Exponencial $e^{-|t|}$', f'E = {E_exp:.2f} J  |  P ≈ 0', 'seagreen'),
          (cosseno, 'Cosseno $\\cos(4\\pi t)$', f'E → ∞  |  P = {P_cos:.3f} W', 'crimson')]

for ax, (s, titulo, info, cor) in zip(axes, sinais):
    ax.fill_between(t, s**2, alpha=0.25, color=cor, label='$|f(t)|^2$')
    ax.plot(t, s, color=cor, lw=1.5, label='$f(t)$')
    ax.set_title(titulo)
    ax.set_xlabel('Tempo (s)')
    ax.set_xlim([-5, 5])
    ax.legend(fontsize=10)
    ax.text(0.5, 0.05, info, transform=ax.transAxes, ha='center',
            fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

axes[0].set_ylabel('Amplitude')
fig.suptitle('Classificação: Sinais de Energia vs. Sinais de Potência', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Pulso retangular : E = {E_pulso:.4f} J  → Sinal de ENERGIA")
print(f"Exponencial      : E = {E_exp:.4f} J  → Sinal de ENERGIA (teórico: 1.0 J)")
print(f"Cosseno (T={T_obs}s): E = {E_cos:.1f} J (cresce com T) → Sinal de POTÊNCIA, P = {P_cos:.4f} W")

---
## 2. Teorema de Parseval — Verificação Numérica

A energia pode ser calculada no tempo **ou** na frequência:

$$E = \int_{-\infty}^{\infty}|f(t)|^2\,dt = \frac{1}{2\pi}\int_{-\infty}^{\infty}|F(\omega)|^2\,d\omega$$

Vamos verificar numericamente para três sinais e visualizar a distribuição de energia.

In [ ]:
N = 8192
fs = 500.0
dt = 1/fs
t = np.arange(-N//2, N//2) * dt
f = fftshift(fftfreq(N, dt))

taus   = [0.5, 1.0, 2.0]  # larguras do pulso
colors = ['royalblue', 'seagreen', 'crimson']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for col, (tau, cor) in enumerate(zip(taus, colors)):
    x = np.where(np.abs(t) <= tau/2, 1.0, 0.0)
    X = fftshift(fft(x) * dt)

    E_t = np.sum(np.abs(x)**2) * dt
    E_f = np.sum(np.abs(X)**2) * (f[1]-f[0])   # integração em Hz (sem 2π no denominador)

    # Domínio do tempo
    axes[0, col].plot(t, x, color=cor, lw=2)
    axes[0, col].fill_between(t, x**2, alpha=0.2, color=cor)
    axes[0, col].set_xlim([-3, 3])
    axes[0, col].set_title(f'τ = {tau} s  →  $E_t$ = {E_t:.3f} J')
    axes[0, col].set_xlabel('Tempo (s)')

    # DEE
    ESD = np.abs(X)**2
    axes[1, col].plot(f, ESD, color=cor, lw=1.5)
    axes[1, col].fill_between(f, ESD, alpha=0.2, color=cor)
    axes[1, col].set_xlim([-8, 8])
    axes[1, col].set_title(f'DEE  →  $E_f$ = {E_f:.3f} J')
    axes[1, col].set_xlabel('Frequência (Hz)')
    axes[1, col].set_ylabel('$|F(f)|^2$  (J/Hz)')

    erro_rel = abs(E_t - E_f) / E_t * 100
    print(f"τ = {tau}s | E_tempo = {E_t:.4f} J | E_freq = {E_f:.4f} J | Erro = {erro_rel:.4f}%")

axes[0, 0].set_ylabel('f(t)')
fig.suptitle('Teorema de Parseval — Energia no Tempo = Energia na Frequência', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nObserve: pulso mais estreito → espectro mais largo (relação inversa tempo-frequência)")

---
## 3. DEE: Comparação entre Sinais

A **Densidade Espectral de Energia** $\Psi(f) = |F(f)|^2$ mostra como a energia se distribui em frequência.

Comparamos três sinais clássicos: pulso retangular, exponencial decrescente e pulso gaussiano.

In [ ]:
N = 16384
fs = 200.0
dt = 1/fs
t = np.arange(-N//2, N//2) * dt
f = fftshift(fftfreq(N, dt))

# Sinais
tau = 1.0;  a = 2.0;  sigma = 0.5
x_rect  = np.where(np.abs(t) <= tau/2, 1.0, 0.0)
x_exp   = np.exp(-a * np.abs(t))
x_gauss = np.exp(-t**2 / (2*sigma**2))

# Normalizamos pela energia total para comparação justa
def dee_norm(x, dt, f):
    X = fftshift(fft(x) * dt)
    ESD = np.abs(X)**2
    E = np.sum(ESD) * (f[1]-f[0])
    return ESD / E  # DEE normalizada

ESD_rect  = dee_norm(x_rect, dt, f)
ESD_exp   = dee_norm(x_exp, dt, f)
ESD_gauss = dee_norm(x_gauss, dt, f)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
sinais_dados = [
    (x_rect,  ESD_rect,  'royalblue',  f'Pulso Retangular\n(τ={tau}s)', r'$\Psi \propto \mathrm{sinc}^2(f\tau)$'),
    (x_exp,   ESD_exp,   'seagreen',   f'Exponencial\n$e^{{-{a}|t|}}$',  r'$\Psi = \frac{4a^2}{(a^2+(2\pi f)^2)^2}$'),
    (x_gauss, ESD_gauss, 'darkorange', f'Gaussiana\n(σ={sigma}s)',       r'$\Psi \propto$ Gaussiana em $f$'),
]

for col, (x, ESD, cor, titulo, formula) in enumerate(sinais_dados):
    # Tempo
    ax_t = axes[0, col]
    ax_t.plot(t, x, color=cor, lw=2)
    ax_t.set_xlim([-4, 4])
    ax_t.set_title(titulo, fontsize=12)
    ax_t.set_xlabel('Tempo (s)')

    # DEE
    ax_f = axes[1, col]
    mask = (f >= -10) & (f <= 10)
    ax_f.fill_between(f[mask], ESD[mask], alpha=0.3, color=cor)
    ax_f.plot(f[mask], ESD[mask], color=cor, lw=1.5)
    ax_f.set_xlabel('Frequência (Hz)')
    ax_f.set_ylabel('DEE normalizada')
    ax_f.set_title(formula, fontsize=11)

    # Linha 3 dB
    pico = ESD[mask].max()
    ax_f.axhline(pico/2, color='gray', ls='--', lw=1, alpha=0.7, label='−3 dB')
    ax_f.legend(fontsize=9)

axes[0, 0].set_ylabel('f(t)')
fig.suptitle('DEE Normalizada — Comparação entre Formas de Pulso', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Largura de Banda que Contém X% da Energia

A **largura de banda de X%** é a menor faixa $[-B_x, B_x]$ tal que:
$$\frac{\int_{-B_x}^{B_x}\Psi(f)\,df}{\int_{-\infty}^{\infty}\Psi(f)\,df} = \alpha$$

Para o pulso retangular de largura τ, calculamos para diferentes porcentagens.

In [ ]:
N = 32768
fs = 500.0
dt = 1/fs
t = np.arange(-N//2, N//2) * dt
f = fftshift(fftfreq(N, dt))
df = f[1] - f[0]

taus = [0.5, 1.0, 2.0]
alphas = [0.50, 0.90, 0.99]
cores  = ['royalblue', 'seagreen', 'crimson']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, tau, cor in zip(axes, taus, cores):
    x = np.where(np.abs(t) <= tau/2, 1.0, 0.0)
    X = fftshift(fft(x) * dt)
    ESD = np.abs(X)**2
    E_total = np.sum(ESD) * df

    # Energia acumulada (do centro para fora)
    f_pos_idx = f >= 0
    f_pos = f[f_pos_idx]
    ESD_pos = ESD[f_pos_idx]
    E_cum = 2 * np.cumsum(ESD_pos) * df  # fator 2: espelho negativo

    mask = (f >= -10) & (f <= 10)
    ax.fill_between(f[mask], ESD[mask]/E_total, alpha=0.15, color=cor)
    ax.plot(f[mask], ESD[mask]/E_total, color=cor, lw=2)

    for alpha in alphas:
        idx = np.searchsorted(E_cum / E_total, alpha)
        if idx < len(f_pos):
            B = f_pos[idx]
            ax.axvline( B, color='gray', ls='--', lw=1.2)
            ax.axvline(-B, color='gray', ls='--', lw=1.2)
            ax.text(B, ax.get_ylim()[1]*0.5 if ax.get_ylim()[1] > 0 else 0.1,
                    f'{int(alpha*100)}%\nB={B:.2f}Hz',
                    fontsize=9, ha='left', color='dimgray')

    ax.set_xlim([-8, 8])
    ax.set_xlabel('Frequência (Hz)')
    ax.set_ylabel('DEE normalizada')
    ax.set_title(f'Pulso retangular τ = {tau} s\n(teórico B_90% = {1/tau:.2f} Hz)')

fig.suptitle('Largura de Banda de X% — Pulso Retangular para Diferentes τ', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Princípio geral: B₉₀% ≈ 1/τ (relação inversa duração × largura de banda)")

---
## 5. Princípio da Incerteza (Heisenberg-Gabor)

$$\Delta t \cdot \Delta f \geq \frac{1}{4\pi}$$

O pulso **Gaussiano** atinge a igualdade. Qualquer outro sinal tem produto $\Delta t \cdot \Delta f$ maior.

Vamos comprimir um pulso Gaussiano e ver como o espectro se alarga.

In [ ]:
N = 16384
fs = 200.0
dt = 1/fs
t = np.arange(-N//2, N//2) * dt
f = fftshift(fftfreq(N, dt))
df_val = f[1]-f[0]

sigmas = [2.0, 1.0, 0.5, 0.25]  # largura temporal (σ)
cores  = ['royalblue', 'seagreen', 'darkorange', 'crimson']

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

produtos = []
for sigma, cor in zip(sigmas, cores):
    x = np.exp(-t**2 / (2*sigma**2))
    E = np.sum(x**2)*dt

    # Δt (desvio padrão temporal)
    Dt = np.sqrt(np.sum(t**2 * x**2)*dt / E)

    X = fftshift(fft(x)*dt)
    ESD = np.abs(X)**2
    E_f = np.sum(ESD)*df_val
    Df = np.sqrt(np.sum(f**2 * ESD)*df_val / E_f)

    produto = Dt * Df
    produtos.append((sigma, Dt, Df, produto))

    mask_t = np.abs(t) <= 6
    mask_f = np.abs(f) <= 6
    label = f'σ={sigma}s'
    ax1.plot(t[mask_t], x[mask_t]/x.max(), color=cor, lw=2, label=label)
    ax2.plot(f[mask_f], ESD[mask_f]/ESD.max(), color=cor, lw=2, label=label)

# Produto Δt·Δf
for sigma, Dt, Df, prod in produtos:
    ax3.bar(f'σ={sigma}', prod, color=cores[sigmas.index(sigma)], alpha=0.7, edgecolor='black')

ax3.axhline(1/(4*np.pi), color='red', ls='--', lw=2, label=f'Mínimo = 1/(4π) ≈ {1/(4*np.pi):.4f}')
ax3.set_ylabel('Δt · Δf')
ax3.set_title('Produto Δt · Δf\n(todos devem estar acima da linha vermelha)')
ax3.legend(fontsize=10)

ax1.set_xlim([-6, 6]); ax1.set_xlabel('Tempo (s)'); ax1.set_ylabel('Amplitude normalizada')
ax1.set_title('Pulso Gaussiano no Tempo'); ax1.legend(fontsize=10)
ax2.set_xlim([-6, 6]); ax2.set_xlabel('Frequência (Hz)'); ax2.set_ylabel('DEE normalizada')
ax2.set_title('DEE do Pulso Gaussiano'); ax2.legend(fontsize=10)

fig.suptitle('Princípio da Incerteza — Pulso mais curto = Espectro mais largo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"{'σ (s)':>8} | {'Δt':>8} | {'Δf':>8} | {'Δt·Δf':>10} | {'Min=1/4π':>10}")
print("-"*55)
for sigma, Dt, Df, prod in produtos:
    print(f"{sigma:>8.2f} | {Dt:>8.4f} | {Df:>8.4f} | {prod:>10.4f} | {1/(4*np.pi):>10.4f}")

---
## 6. PSD de Sinais Periódicos

Para sinais periódicos, a PSD é um espectro **discreto** — impulsos nas frequências harmônicas:

$$S(f) = \sum_{n=-\infty}^{\infty} |c_n|^2\,\delta(f - nf_0)$$

Comparamos senoide pura, onda quadrada e sinal multi-tom.

In [ ]:
fs = 10000
T  = 10.0   # longa janela para boa resolução espectral
t  = np.arange(0, T, 1/fs)
N  = len(t)
f  = np.fft.rfftfreq(N, 1/fs)
f0 = 5.0    # frequência fundamental

def psd_periodograma(x, T, N):
    """PSD via periodograma (|FFT|²/T)"""
    X = np.fft.rfft(x) / fs
    PSD = 2 * np.abs(X)**2 / T  # fator 2: frequências positivas
    return PSD

A = 1.0
senoide   = A * np.cos(2*np.pi*f0*t)
quadrada  = A * sig.square(2*np.pi*f0*t)                # Fourier: c_n = 0 para n par
multitom  = A*np.cos(2*np.pi*f0*t) + 0.5*np.cos(2*np.pi*3*f0*t) + 0.3*np.cos(2*np.pi*5*f0*t)

sinais_p = [
    (senoide,  'Senoide $A\cos(2\pi f_0 t)$',  f'P = {A**2/2:.3f} W (teórico)'),
    (quadrada, 'Onda quadrada',                  f'P = {A**2:.3f} W (teórico)'),
    (multitom, 'Multi-tom (3 harmônicas)',       f'P = {(A**2 + 0.5**2 + 0.3**2)/2:.3f} W (teórico)'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cores = ['royalblue', 'seagreen', 'darkorange']

for col, ((x, titulo, info), cor) in enumerate(zip(sinais_p, cores)):
    PSD = psd_periodograma(x, T, N)
    P_num = np.sum(PSD) * (f[1]-f[0])

    # Tempo (1 período)
    mask_t = t <= 3/f0
    axes[0, col].plot(t[mask_t]*1000, x[mask_t], color=cor, lw=2)
    axes[0, col].set_xlabel('Tempo (ms)')
    axes[0, col].set_title(titulo)
    axes[0, col].set_ylabel('Amplitude (V)')

    # PSD
    mask_f = f <= 8*f0
    axes[1, col].stem(f[mask_f], PSD[mask_f],
                       linefmt=cor+'-', markerfmt='o', basefmt='k-')
    axes[1, col].set_xlabel('Frequência (Hz)')
    axes[1, col].set_ylabel('PSD (W/Hz)')
    axes[1, col].set_title(f'{info}\nP_num = {P_num:.3f} W')
    axes[1, col].set_xlim([-1, 8*f0+1])

axes[0, 0].set_ylabel('Amplitude (V)')
fig.suptitle('PSD de Sinais Periódicos — Espectro Discreto', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Teorema de Wiener-Khinchin

A PSD e a **autocorrelação** são pares de Fourier:

$$R(\tau) \xleftrightarrow{\mathcal{F}} S(f)$$

Demonstramos com uma senoide: autocorrelação é um cosseno, cuja TF são impulsos (PSD).

In [ ]:
fs = 10000
T  = 20.0
t  = np.arange(0, T, 1/fs)
N  = len(t)
f0 = 10.0  # Hz
A  = 2.0

x = A*np.cos(2*np.pi*f0*t) + 0.7*A*np.cos(2*np.pi*3*f0*t)

# Autocorrelação via FFT (método eficiente)
X_fft = np.fft.rfft(x, n=2*N)
Rxx_fft = np.fft.irfft(np.abs(X_fft)**2) / N
Rxx = Rxx_fft[:N//2]
tau = np.arange(len(Rxx)) / fs

# PSD via Wiener-Khinchin (TF da autocorrelação)
S_WK  = np.fft.rfft(Rxx_fft[:N]) / fs
f_WK  = np.fft.rfftfreq(N, 1/fs)

# PSD direta (periodograma)
X_dir = np.fft.rfft(x) / fs
S_dir = 2*np.abs(X_dir)**2 / T
f_dir = np.fft.rfftfreq(N, 1/fs)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Sinal
axes[0].plot(t[:int(3/f0*fs)]*1000, x[:int(3/f0*fs)], 'royalblue', lw=2)
axes[0].set_xlabel('Tempo (ms)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Sinal: 2 tons')

# Autocorrelação
mask_tau = tau <= 0.5
axes[1].plot(tau[mask_tau]*1000, Rxx[mask_tau], 'seagreen', lw=2)
axes[1].set_xlabel('Atraso τ (ms)')
axes[1].set_ylabel('R(τ)')
axes[1].set_title('Autocorrelação R(τ)\n(também periódica!)')
axes[1].axhline(0, color='k', lw=0.8)

# PSD — comparação
mask_f = f_dir <= 6*f0
axes[2].stem(f_dir[mask_f], S_dir[mask_f], linefmt='r-', markerfmt='ro', basefmt='k-', label='Direto (|FFT|²/T)')
axes[2].stem(f_WK[mask_f],  np.abs(S_WK[mask_f])*2/N*fs,
              linefmt='b--', markerfmt='b^', basefmt='k-', label='Wiener-Khinchin')
axes[2].set_xlabel('Frequência (Hz)')
axes[2].set_ylabel('PSD (W/Hz)')
axes[2].set_title('PSD — Comparação dos métodos')
axes[2].legend(fontsize=9)
axes[2].set_xlim([-2, 6*f0+2])

fig.suptitle('Teorema de Wiener-Khinchin:  R(τ) ↔ S(f)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"R(0) = {Rxx[0]:.4f} W  (potência total — confirma P = A²/2 × 2 componentes)")
print(f"P teórica = {A**2/2 + (0.7*A)**2/2:.4f} W")

---
## 8. Ruído Branco, Filtragem e SNR

**Ruído branco** tem PSD constante: $S_n(f) = N_0/2$  
Após um filtro com resposta $H(f)$: $S_y(f) = |H(f)|^2 \cdot S_n(f)$  
Potência de ruído na saída: $P_n = N_0 \cdot B$ (proporcional à largura de banda!)

$$\text{SNR} = \frac{P_s}{N_0 \cdot B}$$

In [ ]:
rng = np.random.default_rng(42)

fs   = 10000
T    = 5.0
N    = int(T * fs)
t    = np.arange(N) / fs
f_ax = np.fft.rfftfreq(N, 1/fs)

N0_2  = 1e-3  # PSD bilateral do ruído (W/Hz)
noise = rng.normal(0, np.sqrt(N0_2 * fs), N)  # ruído branco discreto

# Filtros Butterworth passa-baixa com diferentes bandas
bandas  = [200, 500, 2000]  # Hz
cores_b = ['royalblue', 'seagreen', 'crimson']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

print(f"PSD do ruído branco N₀/2 = {N0_2:.4f} W/Hz")
print(f"{'Banda (Hz)':>12} | {'P_n teórico':>14} | {'P_n numérico':>14} | SNR (dB)")
print("-"*65)

Ps = 0.5  # potência do sinal (senoide A=1)

for col, (B, cor) in enumerate(zip(bandas, cores_b)):
    sos = sig.butter(6, B, btype='low', fs=fs, output='sos')
    y = sig.sosfilt(sos, noise)

    # PSD (Welch para estimativa suave)
    f_w, Pxx_in  = sig.welch(noise, fs, nperseg=2048)
    f_w, Pxx_out = sig.welch(y,     fs, nperseg=2048)

    Pn_teo  = 2 * N0_2 * B
    Pn_num  = np.sum(Pxx_out) * (f_w[1]-f_w[0])
    SNR_dB  = 10*np.log10(Ps / Pn_num)

    # Tempo (primeiros 50 ms)
    mask_t = t <= 0.05
    axes[0, col].plot(t[mask_t]*1000, noise[mask_t], 'lightgray', lw=0.8, alpha=0.8, label='Ruído branco')
    axes[0, col].plot(t[mask_t]*1000, y[mask_t], color=cor, lw=1.5, label=f'Filtrado B={B} Hz')
    axes[0, col].set_xlabel('Tempo (ms)')
    axes[0, col].set_title(f'Saída do filtro (B = {B} Hz)')
    axes[0, col].set_ylabel('Amplitude')
    axes[0, col].legend(fontsize=9)

    # PSD
    axes[1, col].semilogy(f_w, Pxx_in,  'lightgray', lw=1, label='Entrada (branco)')
    axes[1, col].semilogy(f_w, Pxx_out, color=cor,  lw=2,  label=f'Saída')
    axes[1, col].axvline(B, color='black', ls='--', lw=1.2, label=f'fc = {B} Hz')
    axes[1, col].set_xlim([0, fs/2])
    axes[1, col].set_xlabel('Frequência (Hz)')
    axes[1, col].set_ylabel('PSD (W/Hz)')
    axes[1, col].set_title(f'P_n ≈ {Pn_num:.4f} W  |  SNR = {SNR_dB:.1f} dB')
    axes[1, col].legend(fontsize=9)

    print(f"{B:>12} | {Pn_teo:>14.4f} | {Pn_num:>14.4f} | {SNR_dB:>8.1f}")

fig.suptitle('Ruído Branco Filtrado — Potência de Ruído ∝ Largura de Banda', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("\n→ Banda menor = menos ruído = melhor SNR (mas menos sinal também!)")

---
## Resumo — Seções 3.7 e 3.8

| Conceito | Fórmula | Observação |
|---|---|---|
| Energia | $E = \int\|f(t)\|^2\,dt$ | Finita para pulsos |
| Parseval | $E = \int\|F(f)\|^2\,df$ | Conservação entre domínios |
| DEE | $\Psi(f) = \|F(f)\|^2$ | Distribuição espectral de energia |
| Incerteza | $\Delta t \cdot \Delta f \geq 1/(4\pi)$ | Gaussiana atinge igualdade |
| Potência | $P = \lim_{T\to\infty}\frac{1}{T}\int\|f\|^2\,dt$ | Para sinais periódicos |
| PSD | $S(f) = \lim_{T\to\infty}\frac{\|F_T(f)\|^2}{T}$ | Distribuição espectral de potência |
| Wiener-Khinchin | $R(\tau) \leftrightarrow S(f)$ | Par de Fourier |
| LTI | $S_y(f) = \|H(f)\|^2 S_x(f)$ | Igual para DEE e PSD |
| SNR | $P_s/(N_0 B)$ | Aumentar B aumenta ruído! |
